# 1. Mount Google Drive

In [ ]:
# @title 1. Initialize & Connect to Drive (Step 1)🚀
from google.colab import drive

print("🔄 Connecting to Google Drive...")
# This will trigger the authorization popup
drive.mount('/content/drive')
print("✅ Drive Connected! Ready for step 2")

# 2. Download File to Drive

In [ ]:
# @title ⬇️ Download File to Remote Upload
# ==========================================
# 🎨 1. SETTINGS & INPUTS (Use the panel on the right ➡️)
# ==========================================
url = "https://releases.ubuntu.com/24.04.4/ubuntu-24.04.4-live-server-amd64.iso" #@param {type:"string"}
keep_original_name = True #@param {type:"boolean"}
custom_file_name = "renamed_file.iso" #@param {type:"string"}

import os
import requests
import logging
from urllib.parse import urlparse, unquote
from tqdm import tqdm

# Setup logging system for clean output tracking
logging.basicConfig(level=logging.INFO, format='%(message)s')

class Col:
    GREEN = '\033[92m'
    CYAN = '\033[96m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    RESET = '\033[0m'
    BOLD = '\033[1m'

class SecureDownloader:
    def __init__(self, target_url, base_folder="remote upload"):
        self.url = target_url
        self.folder_name = base_folder
        self.base_path = '/content/drive/My Drive'
        self.full_path = os.path.join(self.base_path, self.folder_name)

    def get_safe_filename(self, response, custom_name, keep_original):
        """Determines the filename and sanitizes it to prevent path traversal."""
        if keep_original:
            # Attempt to pull the original name from the server headers
            content_disposition = response.headers.get('content-disposition')
            if content_disposition and 'filename=' in content_disposition:
                fname = content_disposition.split('filename=')[1].strip('"\'')
                return os.path.basename(fname)
            else:
                # Fallback: Extract from the URL string
                parsed_url = urlparse(self.url)
                fname = unquote(os.path.basename(parsed_url.path))
                return fname if fname else "downloaded_file.dat"
        else:
            return os.path.basename(custom_name)

    def setup_directory(self):
        """Prepares the destination folder safely."""
        if not os.path.exists(self.base_path):
            logging.error(f"{Col.RED}❌ Error: Google Drive not mounted! Run Block 1 first.{Col.RESET}")
            return False

        if not os.path.exists(self.full_path):
            os.makedirs(self.full_path)
            logging.info(f"{Col.YELLOW}🆕 Created new folder: {self.folder_name}{Col.RESET}")
        else:
            logging.info(f"{Col.GREEN}✅ Folder found: {self.folder_name}{Col.RESET}")

        os.chdir(self.full_path)
        return True

    def execute_download(self, custom_name, keep_original):
        """Handles the request stream and writes the file."""
        if not self.setup_directory():
            return

        try:
            # Initiate connection with timeout to prevent hanging
            response = requests.get(self.url, stream=True, timeout=30)
            response.raise_for_status()

            final_name = self.get_safe_filename(response, custom_name, keep_original)

            logging.info(f"\n{Col.YELLOW}⬇️ Starting download for: {Col.BOLD}{final_name}{Col.RESET}")

            total_size = int(response.headers.get('content-length', 0))
            block_size = 1024

            # 📊 Progress Bar with Percentage
            with tqdm(total=total_size, unit='iB', unit_scale=True, desc="🚀 Progress", colour='green') as bar:
                with open(final_name, 'wb') as file:
                    for data in response.iter_content(block_size):
                        bar.update(len(data))
                        file.write(data)

            self.report_success(final_name)

        except requests.exceptions.RequestException as e:
            logging.error(f"\n{Col.RED}❌ Network/Download Error: {e}{Col.RESET}")
        except Exception as e:
            logging.error(f"\n{Col.RED}❌ Unexpected System Error: {e}{Col.RESET}")

    def report_success(self, final_name):
        """Validates the downloaded file and reports size."""
        if os.path.exists(final_name):
            file_size_mb = os.path.getsize(final_name) / (1024 * 1024)
            print(f"\n{Col.GREEN}✨ {Col.BOLD}SUCCESS!{Col.RESET}")
            print(f"📄 File: {Col.BOLD}{final_name}{Col.RESET}")
            print(f"📂 Location: {Col.CYAN}My Drive/{self.folder_name}/{Col.RESET}")
            print(f"📦 Size: {Col.BOLD}{file_size_mb:.2f} MB{Col.RESET}")
        else:
            logging.error(f"\n{Col.RED}⚠️ Error: File was not saved to disk.{Col.RESET}")

# ==========================================
# 🚀 2. RUN DOWNLOADER
# ==========================================
downloader = SecureDownloader(url)
downloader.execute_download(custom_file_name, keep_original_name)